# 03 — Overfitting & pruning: depth is the complexity dial

*Notebook 3 of 5 — Decision Trees*

Notebook 02 ended on a clue: the unpruned tree cross-validated a little worse than the small one. Here
we follow that clue. On a curved, noisy dataset we watch a tree go from too-simple to well-fitted to
memorizing, meet the train/test U-curve again, and learn to prune a tree back — choosing how hard to
prune the honest way, with cross-validation.

**Prerequisites**

- Notebooks 01–02 — impurity, growth, the depth-2 tree and its cross-validation hint.
- Module 00 — over- and under-fitting and the generalization gap on `make_moons` (NB 09),
  cross-validation (NB 10), the train/test split (NB 04).

**What you'll be able to do**

- Recognise a tree under- and over-fitting from its decision boundary and its train/test curve.
- Read tree depth as the complexity dial.
- Prune a tree two ways — capping depth (pre-pruning) and cost-complexity (post-pruning).
- Choose how hard to prune with cross-validation, never by peeking at the test set.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import make_moons
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.tree import DecisionTreeClassifier

from ml_course import viz
from ml_course.colors import COLORS

np.random.seed(0)
viz.use_course_style()

# Two interleaving crescents with 30% noise: the curved set a straight line could not fit.
Xm, ym = make_moons(n_samples=300, noise=0.30, random_state=0)
Xtr, Xte, ytr, yte = train_test_split(Xm, ym, test_size=0.30, random_state=0, stratify=ym)
cv = StratifiedKFold(5, shuffle=True, random_state=0)
print(f"train {len(ytr)}, test {len(yte)}")

## Where we are, and why moons

Notebook 02 left a clue: grown to full depth, the tree cross-validated a touch *worse* than the depth-2
one. To see why, we need data a tree can genuinely over-fit. Penguins are too cleanly separable — one
cut nearly solves them. So we switch to `make_moons`: two interleaving crescents with 30% noise, the
curved shape a straight line (logistic regression) could not fit. A tree *can* carve it; the real
question is **how hard to carve** before it starts tracing the noise.

## Depth is the complexity dial

A shallow tree makes a few big boxes. Too shallow and it cannot follow the curve — it **underfits**. A
very deep tree keeps splitting until almost every point sits in its own tiny box — it **overfits**,
drawing fences around random noise. The right model is somewhere between, and **depth** is the dial that
moves us along that range.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
for ax, depth, title in [
    (axes[0], 1, "max_depth = 1 (underfit)"),
    (axes[1], 6, "max_depth = 6"),
    (axes[2], None, "unlimited (overfit)"),
]:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(Xtr, ytr)
    viz.plot_decision_boundary(tree, Xtr, ytr, ax=ax)
    ax.set_title(f"{title} — test {tree.score(Xte, yte):.3f}")
plt.show()

### Read the figure

At `max_depth = 1` the tree makes a single cut — one straight stripe across two curved moons, far too
coarse (it underfits). At `max_depth = 6` the boundary bends with both crescents. Let the tree grow
without limit and it sprouts **thin horizontal and vertical bands** — narrow fences it builds around
individual stray points (an amber point stranded up in the periwinkle zone gets its own slim strip; a
class-0 point lost in the amber gets a vertical sliver near x1 ≈ 1). Those slivers are real decision
regions, not plotting glitches: each one memorizes a single noisy training point — overfitting, drawn.
The test accuracy in each title tells the same story: middling, good, then slightly down.

## The generalization gap, again

Module 00 met this on polynomials; it returns for trees. As we deepen the tree, **training** accuracy
climbs — a deep enough tree memorizes the training set perfectly (zero training error). **Test**
accuracy does not: it rises, peaks, then slips back as the extra depth fits noise the test set does not
share. The widening distance between the two curves is the generalization gap.

In [ ]:
depths = list(range(1, 13))
train_err, test_err, cv_acc = [], [], []
for depth in depths:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(Xtr, ytr)
    train_err.append(1 - tree.score(Xtr, ytr))
    test_err.append(1 - tree.score(Xte, yte))
    clf = DecisionTreeClassifier(max_depth=depth, random_state=0)
    cv_acc.append(cross_val_score(clf, Xtr, ytr, cv=cv).mean())

best_depth = depths[int(np.argmax(cv_acc))]
print("depth :  train_err  test_err   CV")
for depth, tr, te, cm in zip(depths, train_err, test_err, cv_acc, strict=True):
    print(f"  {depth:2d}  :   {tr:.3f}     {te:.3f}    {cm:.4f}")
print(f"CV-best depth = {best_depth}  (CV {max(cv_acc):.4f})")

fig = viz.plot_train_test_curve(depths, train_err, test_err, xlabel="max_depth", ylabel="error")
fig.axes[0].axvline(
    best_depth,
    color=COLORS["highlight"],
    linestyle="--",
    linewidth=1.5,
    label=f"CV-best depth = {best_depth}",
)
fig.axes[0].legend()
plt.show()

### Read the figure

Training error slides all the way to zero by about depth 8 — the tree has memorized every training
point. Test error, in the other colour, falls to its lowest around depth 6–7 and then drifts back up;
the gap between the two curves is the overfitting. The dashed line marks the depth cross-validation will
choose. The U is shallow here because the moons are not very noisy, but the shape — and the
train-error collapse to zero — is the signature to remember.

## Choose the dial by cross-validation, not the test set

The curve above is for *our* understanding. We must not pick a depth by reading the test set — that
would quietly tune the model to it and inflate the score we report (module 00 NB 10). Instead we
cross-validate on the **training** set, let it pick the depth, and read the sealed test exactly
**once**.

In [ ]:
best_tree = DecisionTreeClassifier(max_depth=best_depth, random_state=0).fit(Xtr, ytr)
raw_best_depth = depths[int(np.argmin(test_err))]
print(f"CV chose max_depth = {best_depth} (CV {max(cv_acc):.4f}) using only the training set")
print(f"sealed-test accuracy at depth {best_depth}: {best_tree.score(Xte, yte):.4f}")
peak = 1 - min(test_err)
print(f"(the raw test maximum was depth {raw_best_depth}, test {peak:.4f} -- not chased)")

### Read the result

Cross-validation chose depth 6 using only the training folds, and the sealed test then reports 0.889.
Notice it is *not* the highest test score on the curve (0.900, at depth 7). That is exactly as it should
be: had we picked depth 7 because it topped the test set, the 0.900 would have been an optimistic number
we tuned into existence. The honest estimate is the one cross-validation did not let us cheat on.

## Pruning: grow, then cut back

Capping `max_depth` is **pre-pruning** — we stop the tree early. There is another way: let the tree
grow all the way, then **cut back** the splits that buy too little. That is **cost-complexity pruning**,
controlled by one number, `ccp_alpha`: it charges a penalty for every leaf, so a larger `ccp_alpha`
snips more splits and leaves a smaller tree. At `ccp_alpha = 0` nothing is pruned; raise it and leaves
fall away.

In [ ]:
path = DecisionTreeClassifier(random_state=0).cost_complexity_pruning_path(Xtr, ytr)
alphas = path.ccp_alphas[:-1]  # drop the final alpha, which prunes all the way to the root

leaves, ptest, pcv = [], [], []
for a in alphas:
    tree = DecisionTreeClassifier(random_state=0, ccp_alpha=a).fit(Xtr, ytr)
    leaves.append(tree.get_n_leaves())
    ptest.append(tree.score(Xte, yte))
    clf = DecisionTreeClassifier(random_state=0, ccp_alpha=a)
    pcv.append(cross_val_score(clf, Xtr, ytr, cv=cv).mean())

best_i = int(np.argmax(pcv))
best_alpha = alphas[best_i]
print(f"CV-best ccp_alpha {best_alpha:.5f} -> {leaves[best_i]} leaves, test {ptest[best_i]:.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
ax1.plot(alphas, ptest, color=COLORS["test"], marker="o", linewidth=2)
ax1.axvline(best_alpha, color=COLORS["highlight"], linestyle="--", linewidth=1.5,
            label=f"CV-best = {best_alpha:.4f}")
ax1.set_xlabel("ccp_alpha")
ax1.set_ylabel("test accuracy")
ax1.legend()
ax2.plot(alphas, leaves, color=COLORS["model"], marker="s", linewidth=2)
ax2.axvline(best_alpha, color=COLORS["highlight"], linestyle="--", linewidth=1.5)
ax2.set_xlabel("ccp_alpha")
ax2.set_ylabel("number of leaves")
plt.show()

### Read the figure

Reading left to right as `ccp_alpha` grows: the tree's leaf count (right panel) drops from 23 toward 2,
and its test accuracy (left panel) climbs to a short plateau — best around 8 to 13 leaves — then falls
once we prune away splits that were actually useful. Cross-validation picks the alpha at the 8-leaf end
of that plateau, and the pruned tree's test accuracy, 0.900, is *higher* than the unpruned tree's 0.878.
(Here cross-validation's pick also happens to top the test set — that is this seed being kind; the rule
is still cross-validation on the training set, not the test peak.) Cutting a memorizing tree back made it
generalize better.

In [ ]:
unpruned = DecisionTreeClassifier(random_state=0).fit(Xtr, ytr)
pruned = DecisionTreeClassifier(random_state=0, ccp_alpha=best_alpha).fit(Xtr, ytr)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
viz.plot_decision_boundary(unpruned, Xtr, ytr, ax=ax1)
viz.plot_decision_boundary(pruned, Xtr, ytr, ax=ax2)
ax1.set_title(f"unpruned — {unpruned.get_n_leaves()} leaves, test {unpruned.score(Xte, yte):.3f}")
ax2.set_title(f"CV-pruned — {pruned.get_n_leaves()} leaves, test {pruned.score(Xte, yte):.3f}")
plt.show()

### Read the figure

On the left, the unpruned tree's boundary is ragged — those thin horizontal and vertical bands are the
same fences the unlimited tree drew earlier, 23 leaves each chasing an individual point. They are real
decision regions, not rendering noise: the tree truly carved a narrow box around every stray. On the
right, the CV-pruned tree keeps 8 leaves: the boundary is a clean pair of crescents. Fewer, larger
boxes, and a better test score. Simpler is not a compromise here — it is the improvement.

## What this means

Depth and `ccp_alpha` are two handles on the *same* dial — model complexity. `max_depth` stops the tree
early; `ccp_alpha` grows it then trims it; both are chosen by cross-validation, never on the test set.
One honest note: the test gap here is small (0.889 and 0.900 against 0.878) because the moons are fairly
clean. The lesson is the *shape* — training error collapsing to zero, the U-shaped test curve, the
jagged boundary — not the exact decimals; on messier data the gap is wider and the cost of overfitting
steeper. The node-size knobs (`min_samples_leaf`, `min_samples_split`) and the rest of the estimator are
the next notebook.

## Your turn

1. **Pick a depth.** From the U-curve, name the depth you would ship and say in one sentence why not
   depth 1, and why not depth 12.
2. **Over-prune on purpose.** Refit with `ccp_alpha=0.02` (about 5 leaves), report its train and test
   accuracy, and decide: is this tree under-pruned or over-pruned?
3. **Why zero?** Write the one-line reason a tree's *training* error can always be driven to zero on
   `make_moons`, while its *test* error cannot.

## What you built

- The skill of spotting under- and over-fitting from both a **decision boundary** and a **train/test
  curve**.
- **Depth as the complexity dial**, with the generalization gap drawn for a tree.
- Two ways to **prune**: **pre-pruning** (`max_depth`) and **post-pruning** (cost-complexity,
  `ccp_alpha`).
- Choosing how hard to prune by **cross-validation** — and seeing a pruned tree beat the unpruned one
  on held-out data.

**Vocabulary:** overfitting · underfitting · complexity dial · generalization gap · pre-pruning ·
post-pruning · cost-complexity pruning · `ccp_alpha`.

## Going further (optional)

- **How `ccp_alpha` is defined.** Cost-complexity pruning minimises (training impurity) + α·(number of
  leaves); sweeping α traces the sequence of ever-smaller subtrees the pruning path drew (ESL §9.2.3).
- **Pre- vs post-pruning.** Pre-pruning is cheaper (you never grow the full tree) but can stop too
  early; post-pruning sees the whole tree before deciding. Tuned by cross-validation, either works.

## References

- Breiman, Friedman, Olshen & Stone (1984). *Classification and Regression Trees (CART).* Wadsworth —
  cost-complexity pruning.
- Hastie, Tibshirani & Friedman (2009). *The Elements of Statistical Learning*, §9.2 / §9.2.3.
  DOI: 10.1007/978-0-387-84858-7.
- James, Witten, Hastie & Tibshirani (2021). *An Introduction to Statistical Learning*, §8.1.
  DOI: 10.1007/978-1-0716-1418-1.

*Previous: 02 — Growing a tree, and reading it · Next: 04 — The estimator & its parameters.*